<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>Semantic Networks</font><br>
<font size=20 color=#00c382>Medieval Weapons</font><br>

|**Assessment Code:** |DAT-6002|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S1_26 *CHECK*|
|**Course Tutor:**|Imtiaz ****|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**


---

# Initialise Notebook

In [265]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [266]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime

# Requests
import requests

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
%matplotlib inline

# Colorama
from colorama import Fore, Back, Style

# NetworkX
import networkx as nx

# PyVis
from pyvis.network import Network

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [267]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [268]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [269]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [270]:
internet_required = False

In [271]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Define Semantic Network Class

This class defines a semantic network for any data. The primary purpose is to allow simple gathering of data and to then visualise the network.

In [ ]:
# Define Class SemanticNetwork
class SemanticNetwork:

    # Constructor
    def __init__(self, root_node="RootNode"):

        # Create graph object
        self.G = nx.DiGraph()
        self.root = root_node
        
        # Define palette (from https://coolors.co, using shades option)
        self.palette = {0:"#C3ADFF", 1:"#9670ff", 2:"#7847ff", 3:"#5A1FFF", 4:"#4100F5", 5:"#3100B8", 6:"#22007C"}
        self.bgcolour = "black"
        self.node_text_colour = "white"
        self.edge_label_colour = "#F0D90A"
        self.edge_colour_structural = "#c4830b81"
        self.edge_colour_property = "#7847ff81"

    # Add structural relationships to the network (IS-A type)
    def add_relation(self, child, parent, relation_type="IS-A"):
        self.G.add_edge(child, parent, label=relation_type)

    # Add property relationships to the network (HAS-PART type)
    def add_property(self, node, value, property_name):
        self.G.add_edge(node, value, label=property_name)


    def get_inherited_attribute(self, node, attribute_label):
        """Climbs the IS-A tree to find properties, supporting overrides."""
        current = node
        while current:
            # Check current node for the specific property
            for _, neighbor, data in self.G.out_edges(current, data=True):
                if data.get('label') == attribute_label:
                    return neighbor
            # Climb up the hierarchy
            parents = [target for _, target, data in self.G.out_edges(current, data=True) 
                       if data.get('label') in ["IS-A", "INSTANCE-OF"]]
            current = parents[0] if parents else None
        return "N/A"
    

    # Calculates distance from the root node
    # Used for dynamic scaling and chosing colours from palette
    def get_node_level(self, node):
        try:
            return nx.shortest_path_length(self.G.to_undirected(), source=node, target=self.root)
        except:
            return None

    # Export Network to json file 
    def export_to_json(self, filename="network.json"):
        data = nx.node_link_data(self.G)

        # *** Note: WRAP WITH A TRY 
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)

        print_output(f"Successfully saved", f"{len(self.G.nodes)} nodes to {filename}")

    # Import Network from json file 
    def import_from_json(self, filename="network.json"):
        
        # *** Note: WRAP WITH A TRY 
        with open(filename, 'r') as f:
            data = json.load(f)

        # Destructive
        self.G = nx.node_link_graph(data)
        print_output(f"Successfully loaded", f"{len(self.G.nodes)} nodes from {filename}")


    # Create a dynamic visualisation of the network using PyVis, saved as HTML file
    def draw(self, filename="dynamic_network_diagram"):

        # Initiate the PyVis Network
        net = Network(height="1000px", width="100%", bgcolor=self.bgcolour, font_color=self.node_text_colour, directed=True)
        
        dist = {n: self.get_node_level(n) for n in self.G.nodes() if self.get_node_level(n) is not None}
        
        # Add Nodes 
        for node in self.G.nodes():
            level = dist.get(node, 99)
            color = self.palette.get(level, "#D3D3D3")
            
            # Dynamic Scaling of nodes, but font doesn't seem to work...
            if level==0:
                dynamic_node_size = 120
                dynamic_font_size = 120
            else:
                dynamic_font_size = max(80 - (level * 8), 25) if level != 99 else 20
                dynamic_node_size = max(70 - (level * 7), 15) if level != 99 else 10
            
            net.add_node(
                node, 
                label=node, 
                color=color, 
                #size=50 if level == 0 else 20,
                size = dynamic_node_size,
                font={'size': dynamic_font_size, 'face': 'arial', 'color': 'white'}
            )

        # Add Edges
        for source, target, data in self.G.edges(data=True):
            label = data.get('label')
            is_structural = label in ["IS-A", "INSTANCE-OF"]
            net.add_edge(
                source, 
                target, 
                label=label,
                width=5 if is_structural else 2, # Extra thick to prove it works
                color= self.edge_colour_structural if is_structural else self.edge_colour_property,
                font={'color': self.edge_label_colour, 'size': 20, 'strokeWidth': 0}
            )

        # Define options for network
        options = {
            "nodes": {
                "font": {
                    "size": 30 
                }
            },
            "edges": {
                "font": {
                    "size": 18,
                    "strokeWidth": 0
                },
                "color": {"inherit": False}
            },
            "physics": {
                "enabled": True,
                "barnesHut": {
                    "gravitationalConstant": -13000, # Push nodes apart
                    "springLength": 300 # Lengthen the edges
                },
                "solver": "barnesHut"
            },
            "interaction": {
                "hideEdgesOnDrag": False,
                "hideEdgesOnZoom": False,
                "zoomView": True
            }
        }
        
        # Convert options for the visualisation JavaScript
        net.set_options(json.dumps(options))

        # Write out the diagram to the specified file
        net.write_html(filename)


    def generate_report(self):
        report_data = []
        # Filter for 'Concept' nodes (those with a hierarchical level)
        concept_nodes = [n for n in self.G.nodes() if self.get_node_level(n) is not None]
        
        for node in concept_nodes:
            report_data.append({
                "Entity": node,
                "Level": self.get_node_level(node),
                "Damage": self.get_inherited_attribute(node, "causesDamage"),
                "Material": self.get_inherited_attribute(node, "primaryMaterial"),
                "Origin": self.get_inherited_attribute(node, "historicalOrigin"),
                "Era": self.get_inherited_attribute(node, "eraUsed")
            })
        return pd.DataFrame(report_data).sort_values("Level")


# Populate Network

In [273]:
# Create new network object (Level 0)
weapons_net = SemanticNetwork(root_node="Weapon")

# Add weapons to the hierarchy

### Level 1
weapons_net.add_relation("Melee", "Weapon")
weapons_net.add_relation("Ranged", "Weapon")
weapons_net.add_relation("Siege Weapon", "Weapon")

### Level 2
# Melee Weapon Categories
weapons_net.add_relation("Bladed Hand Weapon", "Melee")
weapons_net.add_relation("Blunt Hand Weapon", "Melee")
weapons_net.add_relation("Polearm", "Melee")

# Ranged Weapon Categories
weapons_net.add_relation("Francisca", "Ranged")
weapons_net.add_relation("Javelin", "Ranged")
weapons_net.add_relation("Bow", "Ranged")
weapons_net.add_relation("Arbalest", "Ranged")
weapons_net.add_relation("Gun", "Ranged")
weapons_net.add_relation("Artillery", "Ranged")

# Siege Weapon Categories
weapons_net.add_relation("Siege Tower", "Siege Weapon")
weapons_net.add_relation("Battering Ram", "Siege Weapon")
weapons_net.add_relation("Cat", "Siege Weapon")
weapons_net.add_relation("Weasel", "Siege Weapon")
weapons_net.add_relation("Chemical Weapon", "Siege Weapon")
weapons_net.add_relation("Biological Weapon", "Siege Weapon")
weapons_net.add_relation("Psychological Weapon", "Siege Weapon")


### Level 3

# Bladed Hand Weapon Categories
weapons_net.add_relation("Sword", "Bladed Hand Weapon")
weapons_net.add_relation("Dagger", "Bladed Hand Weapon")
weapons_net.add_relation("Knife", "Bladed Hand Weapon")

# Blunt Melee Weapon Categories
weapons_net.add_relation("Club", "Blunt Hand Weapon")
weapons_net.add_relation("Mace", "Blunt Hand Weapon")
weapons_net.add_relation("Morning Star", "Blunt Hand Weapon")
weapons_net.add_relation("Holy Water Sprinkler", "Blunt Hand Weapon")
weapons_net.add_relation("Flail", "Blunt Hand Weapon")
weapons_net.add_relation("War Hammer", "Blunt Hand Weapon")
weapons_net.add_relation("Horseman's Pick", "Blunt Hand Weapon")

# Polearm Categories
weapons_net.add_relation("Quarterstaff", "Polearm")
weapons_net.add_relation("Spear", "Polearm")
weapons_net.add_relation("Winged Spear", "Polearm")
weapons_net.add_relation("Lance", "Polearm")
weapons_net.add_relation("Pike", "Polearm")
weapons_net.add_relation("Corseque", "Polearm")
weapons_net.add_relation("Fauchard", "Polearm")
weapons_net.add_relation("Glaive", "Polearm")
weapons_net.add_relation("Guisarme", "Polearm")
weapons_net.add_relation("Halberd", "Polearm")
weapons_net.add_relation("Danish Axe", "Polearm")
weapons_net.add_relation("Sparth", "Polearm")
weapons_net.add_relation("Bardiche", "Polearm")
weapons_net.add_relation("Pollaxe", "Polearm")
weapons_net.add_relation("Modern Flanged Mace", "Polearm")
weapons_net.add_relation("Maul", "Polearm")
weapons_net.add_relation("Becs de Corbin", "Polearm")

# Bow Categories
weapons_net.add_relation("Cross Bow", "Bow")
weapons_net.add_relation("Long Bow", "Bow")

# Guns and Artillery Categories
weapons_net.add_relation("Hand Cannon", "Gun")
weapons_net.add_relation("Arquebus", "Gun")

weapons_net.add_relation("Stone Thrower (Pierrier)", "Artillery")
weapons_net.add_relation("Cannon", "Artillery")


### Level 4
# Swords
weapons_net.add_relation("Arming Sword", "Sword")
weapons_net.add_relation("Broad Sword", "Sword")
weapons_net.add_relation("Falchion", "Sword")
weapons_net.add_relation("Long Sword", "Sword")

# Dagger and Knives
weapons_net.add_relation("Anelace", "Dagger") 
weapons_net.add_relation("Stiletto", "Dagger") 
weapons_net.add_relation("Poingnard", "Dagger") 
weapons_net.add_relation("Rondel", "Dagger") 



# Artillery
weapons_net.add_relation("Traction Trebuchet","Stone Thrower (Pierrier)")
weapons_net.add_relation("Counterweight Trebuchet","Stone Thrower (Pierrier)")
weapons_net.add_relation("Onager","Stone Thrower (Pierrier)")
weapons_net.add_relation("Mangonel","Stone Thrower (Pierrier)")
weapons_net.add_relation("Traction Trebuchet","Stone Thrower (Pierrier)")
weapons_net.add_relation("Ballista","Stone Thrower (Pierrier)")
weapons_net.add_relation("Springald","Stone Thrower (Pierrier)")

weapons_net.add_relation("Bombard","Cannon")
weapons_net.add_relation("Petard","Cannon")

#weapons_net.add_relation("Colonel Blood's Dagger", "Stiletto", "INSTANCE-OF") # Level 5

# Level 5
# Define Properties and override where necessary

weapons_net.add_property("Bladed Hand Weapon", "Steel", "primaryMaterial")

weapons_net.add_property("Dagger", "Puncture", "causesDamage")
weapons_net.add_property("Knife", "Cut", "causesDamage")
weapons_net.add_property("Arming Sword", "Cut", "causesDamage")
weapons_net.add_property("Arming Sword", "Bash", "causesDamage")
weapons_net.add_property("Arming Sword", "Puncture", "causesDamage")
weapons_net.add_property("Arming Sword", "Pommel", "hasPart")
weapons_net.add_property("Arming Sword", "Hilt", "hasPart")

weapons_net.add_property("Stiletto", "Puncture", "causesDamage") 
weapons_net.add_property("Stiletto", "15th Century", "eraUsed")

weapons_net.add_property("Blunt Hand Weapon", "Bash", "causesDamage")
weapons_net.add_property("Blunt Hand Weapon", "Wood", "primaryMaterial")
weapons_net.add_property("Mace", "Puncture", "causesDamage")


# Generate Visualisations

In [ ]:
print(weapons_net.generate_report())

# Create the interactive file
filename = "medieval_weapons_semantic_network_" + now_datetime_timestamp + ".html"
weapons_net.draw(filename)

                Entity  Level Damage Material Origin  Era
1               Weapon      0    N/A      N/A    N/A  N/A
0                Melee      1    N/A      N/A    N/A  N/A
2               Ranged      1    N/A      N/A    N/A  N/A
3         Siege Weapon      1    N/A      N/A    N/A  N/A
4   Bladed Hand Weapon      2    N/A    Steel    N/A  N/A
..                 ...    ...    ...      ...    ...  ...
71                 Cut      4    N/A      N/A    N/A  N/A
70            Puncture      4    N/A      N/A    N/A  N/A
74                Hilt      5    N/A      N/A    N/A  N/A
73              Pommel      5    N/A      N/A    N/A  N/A
75        15th Century      5    N/A      N/A    N/A  N/A

[77 rows x 6 columns]


In [ ]:
# Save the network out
filename = "medieval_weapons_semantic_network_" + now_datetime_timestamp + ".json"
weapons_net.export_to_json(filename)

# Import from json
#weapons_net.import_from_json(filename)

Successfully saved: 77 nodes to medieval_weapons_semantic_network20260216-2303.json
